In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os   
import cv2
from pathlib import Path
import random

In [ ]:
data = "data/recodai-luc-scientific-image-forgery-detection/train_images"
masks = "data/recodai-luc-scientific-image-forgery-detection/train_masks"
categories = ["forged", "authentic"]

dataset = []

for category in categories:
    path = os.path.join(data, category)

    if category == "forged":
        img_files = sorted(
            f for f in os.listdir(path)
            if os.path.isfile(os.path.join(path, f))
        )
        mask_files = sorted(
            f for f in os.listdir(masks)
            if os.path.isfile(os.path.join(masks, f))
        )

        for img, mask in zip(img_files, mask_files):
            if Path(img).stem != Path(mask).stem:
                print("Mismatch:", img, mask)
                continue

            img_array = cv2.imread(
                os.path.join(path, img),
                cv2.IMREAD_GRAYSCALE
            )

            mask_array = np.load(os.path.join(masks, mask))
            if mask_array.ndim == 3:
                mask_array = np.any(mask_array > 0, axis=0).astype(np.float32)
            else:
                mask_array = (mask_array > 0).astype(np.float32)

            new_img_array = cv2.resize(img_array, (512, 512))
            new_mask_array = cv2.resize(
                mask_array, (512, 512), interpolation=cv2.INTER_NEAREST
            )
            new_mask_array = (new_mask_array > 0).astype(np.float32)

            new_img_array = np.expand_dims(new_img_array, axis=-1)
            new_mask_array = np.expand_dims(new_mask_array, axis=-1)

            dataset.append([new_img_array, new_mask_array])

    else:  # authentic
        img_files = sorted(
            f for f in os.listdir(path)
            if os.path.isfile(os.path.join(path, f))
        )

        for img in img_files:
            img_array = cv2.imread(
                os.path.join(path, img),
                cv2.IMREAD_GRAYSCALE
            )

            new_img_array = cv2.resize(img_array, (512, 512))
            new_img_array = np.expand_dims(new_img_array, axis=-1)

            zero_mask = np.zeros((512, 512, 1), dtype=np.float32)

            dataset.append([new_img_array, zero_mask])

In [15]:
random.shuffle(dataset)

In [1]:
X = []
y = []
for feature, label in dataset:
    X.append(feature)
    y.append(label)
y = np.expand_dims(y, axis=-1)

NameError: name 'dataset' is not defined

In [6]:
y.shape

(2751, 512, 512, 1)

In [7]:
X = np.array(X)

In [8]:
X.shape

(2751, 512, 512, 1)

In [9]:
np.save("X.npy", X)
np.save("y.npy", y)

In [10]:
X = np.load("X.npy")

In [11]:
X = X.astype(np.float32) / 255.0

In [12]:
np.save("X_norm.npy", X)